In [1]:
!pip install torch numpy matplotlib


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# --- Config ---
ALPHA      = 1.0
T_MAX      = 0.01
N_DOMAIN   = 10000
N_BOUNDARY = 800
N_INITIAL  = 500
W_PDE = 1
W_BC  = 10
W_IC  = 10
T_EVAL     = [0.001, 0.005, 0.010]
LX_RANGE   = [0.5, 2.0]
LY_RANGE   = [0.5, 2.0]

# Fourier feature config — same bands as phase 2
SIGMA_BANDS = [1, 3, 5]
M_FREQ      = 32  # random frequencies per band; total Fourier features = 3×2×32 = 192

N_ADAM  = 10000
N_LBFGS = 500

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print(f"Device: {DEVICE}")
print(f"Fourier features: {len(SIGMA_BANDS)} bands × 2 × {M_FREQ} = {len(SIGMA_BANDS)*2*M_FREQ}")
print(f"MLP input size: {len(SIGMA_BANDS)*2*M_FREQ + 2}  (192 Fourier + 2 geometry)")
print("Config loaded.")

Device: cpu
Fourier features: 3 bands × 2 × 32 = 192
MLP input size: 194  (192 Fourier + 2 geometry)
Config loaded.


In [3]:
def T_exact(x, y, t, Lx, Ly, alpha=ALPHA):
    """
    Two-mode exact solution on [0,Lx]×[0,Ly]:
      mode 1: sin(πx/Lx)sin(πy/Ly) · exp(−λ₁t)
      mode 2: 0.3·sin(5πx/Lx)sin(5πy/Ly) · exp(−λ₂t)
    λ₁ = π²α(1/Lx² + 1/Ly²),  λ₂ = 25λ₁
    """
    lam1 = np.pi**2 * alpha * (1/Lx**2 + 1/Ly**2)
    lam2 = 25 * lam1
    low  = np.sin(np.pi * x / Lx) * np.sin(np.pi * y / Ly) * np.exp(-lam1 * t)
    high = 0.3 * np.sin(5*np.pi*x/Lx) * np.sin(5*np.pi*y/Ly) * np.exp(-lam2 * t)
    return low + high

# Sanity check 1: unit-square case matches phase 2 at t=0 (IC)
x0, y0 = np.array([0.5]), np.array([0.3])
val = T_exact(x0, y0, 0.0, 1.0, 1.0)
ic  = np.sin(np.pi*x0)*np.sin(np.pi*y0) + 0.3*np.sin(5*np.pi*x0)*np.sin(5*np.pi*y0)
assert np.isclose(val, ic).all(), f"Unit-square IC mismatch: {val} vs {ic}"
print(f"Sanity 1 passed: T_exact(0.5,0.3,0,1,1) = {val[0]:.6f}")

# Sanity check 2: BC is zero at x=0 for any Lx, Ly, t
val_bc = T_exact(np.array([0.0]), y0, 0.005, 1.5, 0.8)
assert np.isclose(val_bc, 0.0).all(), f"BC not zero: {val_bc}"
print(f"Sanity 2 passed: T_exact(0,y,t,Lx,Ly) = {val_bc[0]:.6f} (should be 0)")

# Sanity check 3: larger plate decays more slowly
lam1_small = np.pi**2 * ALPHA * (1/0.5**2 + 1/0.5**2)
lam1_large = np.pi**2 * ALPHA * (1/2.0**2 + 1/2.0**2)
assert lam1_small > lam1_large, "Small plate should decay faster"
print(f"Sanity 3 passed: λ₁(0.5×0.5)={lam1_small:.2f} > λ₁(2×2)={lam1_large:.2f}")

print("\nDecay rates at unit square (Lx=Ly=1):")
lam1 = np.pi**2 * ALPHA * 2
print(f"  λ₁ = {lam1:.4f}  (mode 1 half-life = {np.log(2)/lam1*1000:.2f} ms)")
print(f"  λ₂ = {25*lam1:.4f}  (mode 2 half-life = {np.log(2)/(25*lam1)*1000:.2f} ms)")

Sanity 1 passed: T_exact(0.5,0.3,0,1,1) = 0.509017
Sanity 2 passed: T_exact(0,y,t,Lx,Ly) = 0.000000 (should be 0)
Sanity 3 passed: λ₁(0.5×0.5)=78.96 > λ₁(2×2)=4.93

Decay rates at unit square (Lx=Ly=1):
  λ₁ = 19.7392  (mode 1 half-life = 35.12 ms)
  λ₂ = 493.4802  (mode 2 half-life = 1.40 ms)


In [4]:
class ParametricFourierNet(nn.Module):
    """
    5D input: (x, y, t, Lx, Ly) in physical coordinates.
    Inside forward():
      - Normalize (x,y,t,Lx,Ly) to [-1,1]
      - Apply multi-scale Fourier features to (xi_norm, eta_norm, t_norm)
      - Concatenate (Lx_norm, Ly_norm) directly → 194-dim MLP input
    PyTorch autograd differentiates through the normalization automatically,
    so PDE residuals use standard ∂T/∂x, ∂T/∂y, ∂T/∂t.
    """
    def __init__(self, sigma_bands, m_freq, mlp_hidden, t_max):
        super().__init__()
        self.sigma_bands = sigma_bands
        self.t_max = t_max
        n_fourier = len(sigma_bands) * 2 * m_freq

        # Frozen random frequency matrices, one per band; shape (3, m_freq)
        for i, sigma in enumerate(sigma_bands):
            B = torch.randn(3, m_freq) * sigma
            self.register_parameter(f"B_{i}", nn.Parameter(B, requires_grad=False))

        # MLP: (n_fourier + 2) → hidden layers → 1
        sizes = [n_fourier + 2] + mlp_hidden + [1]
        layers = []
        for in_sz, out_sz in zip(sizes[:-2], sizes[1:-1]):
            layers += [nn.Linear(in_sz, out_sz), nn.Tanh()]
        layers.append(nn.Linear(sizes[-2], sizes[-1]))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        # x: (N, 5) — columns: [x_phys, y_phys, t, Lx, Ly]
        lx = x[:, 3:4]  # (N, 1)
        ly = x[:, 4:5]  # (N, 1)

        # Normalize physical coords to [-1, 1]
        xi_n  = 2.0 * x[:, 0:1] / lx - 1.0          # x/Lx rescaled
        eta_n = 2.0 * x[:, 1:2] / ly - 1.0           # y/Ly rescaled
        t_n   = 2.0 * x[:, 2:3] / self.t_max - 1.0   # t/T_MAX rescaled
        lx_n  = (lx - 1.25) / 0.75                    # Lx ∈ [0.5,2.0] → [-1,1]
        ly_n  = (ly - 1.25) / 0.75                    # Ly ∈ [0.5,2.0] → [-1,1]

        coords = torch.cat([xi_n, eta_n, t_n], dim=1)  # (N, 3) for Fourier

        # Multi-scale Fourier features: sin and cos projections per band
        features = []
        for i in range(len(self.sigma_bands)):
            B = getattr(self, f"B_{i}")   # (3, m_freq)
            proj = coords @ B              # (N, m_freq)
            features.append(torch.sin(proj))
            features.append(torch.cos(proj))

        # Concat Fourier features (192-dim) + geometry params (2-dim) = 194-dim
        z = torch.cat(features + [lx_n, ly_n], dim=-1)
        return self.mlp(z)


# Shape test
_net = ParametricFourierNet(SIGMA_BANDS, M_FREQ, [128, 128, 128], T_MAX).to(DEVICE)
_x   = torch.rand(16, 5, device=DEVICE)
_x[:, 3] = 1.0   # Lx = 1.0
_x[:, 4] = 1.0   # Ly = 1.0
_x[:, 0] *= 1.0  # x ∈ [0, Lx]
_x[:, 1] *= 1.0  # y ∈ [0, Ly]
_x[:, 2] *= T_MAX
out = _net(_x)
assert out.shape == (16, 1), f"Expected (16,1), got {out.shape}"
print(f"Shape test passed: (16,5) → {out.shape}")

total_p    = sum(p.numel() for p in _net.parameters())
trainable  = sum(p.numel() for p in _net.parameters() if p.requires_grad)
print(f"Total parameters : {total_p:,}")
print(f"Trainable params : {trainable:,}  (frozen Fourier matrices excluded)")
del _net, _x, out

Shape test passed: (16,5) → torch.Size([16, 1])
Total parameters : 58,401
Trainable params : 58,113  (frozen Fourier matrices excluded)


In [5]:
def sample_interior(Lx, Ly, n, device):
    """N interior points in [0,Lx]×[0,Ly]×[0,T_MAX]. requires_grad=True for autograd."""
    x  = torch.rand(n, 1, device=device) * Lx
    y  = torch.rand(n, 1, device=device) * Ly
    t  = torch.rand(n, 1, device=device) * T_MAX
    lx = torch.full((n, 1), Lx, device=device)
    ly = torch.full((n, 1), Ly, device=device)
    pts = torch.cat([x, y, t, lx, ly], dim=1)
    pts.requires_grad_(True)
    return pts

def sample_boundary(Lx, Ly, n, device):
    """N boundary points on the four edges, t ~ U[0, T_MAX]. No requires_grad needed."""
    q = n // 4
    edges = []
    for x_val in [0.0, Lx]:
        y  = torch.rand(q, 1, device=device) * Ly
        t  = torch.rand(q, 1, device=device) * T_MAX
        xc = torch.full((q, 1), x_val, device=device)
        edges.append(torch.cat([xc, y, t,
                                 torch.full((q, 1), Lx, device=device),
                                 torch.full((q, 1), Ly, device=device)], dim=1))
    for y_val in [0.0, Ly]:
        x  = torch.rand(q, 1, device=device) * Lx
        t  = torch.rand(q, 1, device=device) * T_MAX
        yc = torch.full((q, 1), y_val, device=device)
        edges.append(torch.cat([x, yc, t,
                                 torch.full((q, 1), Lx, device=device),
                                 torch.full((q, 1), Ly, device=device)], dim=1))
    return torch.cat(edges, dim=0)

def sample_ic(Lx, Ly, n, device):
    """N IC points: x∈[0,Lx], y∈[0,Ly], t=0."""
    x  = torch.rand(n, 1, device=device) * Lx
    y  = torch.rand(n, 1, device=device) * Ly
    t  = torch.zeros(n, 1, device=device)
    lx = torch.full((n, 1), Lx, device=device)
    ly = torch.full((n, 1), Ly, device=device)
    return torch.cat([x, y, t, lx, ly], dim=1)

def ic_target(pts):
    """IC values for points (N,5). Uses columns 0,1,3,4 for x,y,Lx,Ly."""
    x, y, lx, ly = pts[:,0:1], pts[:,1:2], pts[:,3:4], pts[:,4:5]
    low  = torch.sin(np.pi * x / lx) * torch.sin(np.pi * y / ly)
    high = 0.3 * torch.sin(5*np.pi*x/lx) * torch.sin(5*np.pi*y/ly)
    return low + high

def pde_residual(model, pts_pde):
    """
    PDE residual: ∂T/∂t − α(∂²T/∂x² + ∂²T/∂y²)
    pts_pde must have requires_grad=True (produced by sample_interior).
    """
    T = model(pts_pde)                                                     # (N,1)
    g1 = torch.autograd.grad(T.sum(), pts_pde, create_graph=True)[0]      # (N,5)
    T_x = g1[:, 0:1]
    T_y = g1[:, 1:2]
    T_t = g1[:, 2:3]
    T_xx = torch.autograd.grad(T_x.sum(), pts_pde, create_graph=True)[0][:, 0:1]
    T_yy = torch.autograd.grad(T_y.sum(), pts_pde, create_graph=True)[0][:, 1:2]
    return T_t - ALPHA * (T_xx + T_yy)

# Sampler shape tests
_Lx, _Ly = 1.5, 0.8
_pde = sample_interior(_Lx, _Ly, 10, DEVICE)
_bc  = sample_boundary(_Lx, _Ly, 20, DEVICE)
_ic  = sample_ic(_Lx, _Ly, 8, DEVICE)

assert _pde.shape == (10, 5) and _pde.requires_grad
assert _bc.shape  == (20, 5)
assert _ic.shape  == (8,  5)

# Boundary points should have x∈{0,Lx} or y∈{0,Ly}
x_bc = _bc[:, 0].cpu(); y_bc = _bc[:, 1].cpu()
on_boundary = ((x_bc == 0) | (x_bc == _Lx) | (y_bc == 0) | (y_bc == _Ly))
assert on_boundary.all(), "Some BC points are not on the boundary"

# IC points should have t=0
assert (_ic[:, 2] == 0).all(), "IC points must have t=0"

print("Sampler tests passed.")
print(f"  Interior: {_pde.shape}, requires_grad={_pde.requires_grad}")
print(f"  Boundary: {_bc.shape}")
print(f"  IC:       {_ic.shape}")
del _pde, _bc, _ic

Sampler tests passed.
  Interior: torch.Size([10, 5]), requires_grad=True
  Boundary: torch.Size([20, 5])
  IC:       torch.Size([8, 5])
